In [ ]:
# ==========================================
# DESCRIPTION STYLE SELECTION
# ==========================================

def get_style(row):

    if str(row.get("Mountain View", "")).lower() in ["yes", "1", "true"]:
        return "amenity-focused"

    elif row.get("town_proximity", "") == "Very Close":
        return "accessibility-focused"

    elif float(row.get("rating", 0)) >= 4.5:
        return "rating-focused"

    return "location-focused"

In [ ]:
# ==========================================
# PROMPT GENERATION
# ==========================================

def create_prompt(row):

    amenities = []

    amenity_columns = [
        "Wifi",
        "Parking",
        "Breakfast",
        "Mountain View",
        "Room Service",
        "Bonfire/Barbeque",
        "Pickup and Dropoff Service"
    ]

    for col in amenity_columns:

        value = str(
            row.get(col, "")
        ).strip().lower()

        if value in ["yes", "1", "true"]:
            amenities.append(col)

    amenities_text = ", ".join(amenities)

    return f"""
Generate a tourism-friendly homestay description.

Writing Style:
{get_style(row)}

Rules:
- Use only the supplied information.
- Do not invent facts.
- Do not mention facilities that are not listed.
- Do not mention mountain views unless available.
- Write between 50 and 80 words.
- Return only the description.

Homestay Name:
{row.get('Name of the Home stay', '')}

Village:
{row.get('Village', '')}

Block:
{row.get('Block', '')}

Category:
{row.get('Category', '')}

Rating:
{row.get('rating', '')}

Review Count:
{row.get('review_count', '')}

Town Proximity:
{row.get('town_proximity', '')}

Deolo Proximity:
{row.get('deolo_proximity', '')}

Durpin Proximity:
{row.get('durpin_proximity', '')}

Amenities:
{amenities_text}
"""

In [ ]:
# ==========================================
# QWEN DESCRIPTION GENERATION
# ==========================================

MODEL_NAME = "qwen3:4b"

descriptions = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    prompt = create_prompt(row)

    response = chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    descriptions.append(
        response["message"]["content"]
        .strip()
    )

df["Description"] = descriptions